# 實驗設計與重現

對應 DLI：02–05。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/09_experimental_design.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/09-experimental-design.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

## 學習目標與先備知識

先讀資料切分及模型評估。學完能用表格寫出「改了什麼、固定什麼、量什麼」，並區分重跑一次與可重現的比較。



## 1. 問題要先寫成可檢驗的假設

**來源摘要 [O9](https://openstax.org/books/introductory-statistics-2e/pages/1-4-experimental-design-and-ethics)：**實驗設計區分解釋變數與反應變數，並考慮其他變數的干擾。觀察到一起變動，不足以直接判定因果。

以下全部為本教材自行設計的 AI 實驗延伸，不是來源的原例題。

問題「CNN 好不好？」太廣。可改成：「在相同 ASL 切分、相同訓練預算下，加入預先定義的資料增強，是否改善未見拍攝者的 Macro-F1？」

| 設計項目 | 本例內容 |
|---|---|
| 操作因素 | 是否使用指定資料增強 |
| 反應指標 | Macro-F1、每類 Recall、耗時 |
| 固定條件 | 資料切分、模型架構、訓練預算、種子集合 |
| 可能干擾 | 不同調參次數、不同資料版本或前處理 |
| 適用範圍 | 本資料、設備與拍攝者設定 |



## 2. 為什麼一次改一件事？

A 用小模型、少資料、無增強；B 用大模型、多資料、有增強。若 B 較好，只能說整套方案在本次評估較好，不能把提升全部歸給增強。

入門可先固定其他條件，一次改一個因素。若想知道因素之間是否有交互作用，還需更完整的多因素設計；一次改一件事不是所有研究的唯一方法。



## 3. 相同種子集合的配對比較

本教材自編五次驗證 Accuracy：

| 種子 | A | B | B−A |
|---:|---:|---:|---:|
| 0 | 0.80 | 0.82 | 0.02 |
| 1 | 0.82 | 0.84 | 0.02 |
| 2 | 0.81 | 0.80 | −0.01 |
| 3 | 0.83 | 0.85 | 0.02 |
| 4 | 0.79 | 0.81 | 0.02 |

A 平均 0.810，B 平均 0.824，差 0.014，即 1.4 個百分點。第三次 B 反而下降，所以不能只挑 B 最好的那一次報告。

樣本標準差可描述這五次結果的分散，但不是自動得到母體效果的信賴區間。相同種子是控制隨機條件的一種方式，不保證不同演算法內部使用完全相同的隨機過程。



## 4. 可重現需要記錄什麼？

| 類別 | 至少記錄 |
|---|---|
| 資料 | 來源、版本、前處理、切分索引或群組 |
| 環境 | Python、套件、硬體及必要設定 |
| 訓練 | 架構、超參數、種子、停止規則 |
| 評估 | 正類定義、指標、閾值、各次結果 |
| 選擇 | 試過幾個方案、用什麼資料選最好者 |

同一種子在不同平台、版本或非決定性 GPU 運算下，未必逐位元相同。因此報告需保留版本與合理的數值容忍度。



## 5. 把測試留到最後

本課程式固定資料切分，使用五個種子比較 RandomForest 的兩種深度，只讀取驗證分數。X_test、y_test 已封存，尚未拿來挑設定。

若後續用驗證結果選定深度，就先固定選擇流程，再最後測一次。大量嘗試仍可能過度適應驗證集，需考慮另設資料或巢狀評估。本練習沒有自動完成所有研究設計問題。

## 自我檢核

1. B 比 A 快，但使用不同 GPU，能說演算法一定更有效率嗎？
2. 五次平均提高是否足以保證所有部署場景都提高？
3. 只公布五十次訓練中最好的那次，有何問題？

### 參考解答

1. 不能直接歸因；硬體是可能干擾，應固定或分別報告。
2. 不足以；資料代表性及部署條件仍限制結論。
3. 選擇性報告會過度樂觀，應呈現預先定義的比較及所有相關結果。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X, y = load_digits(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, stratify=y_dev, random_state=43)
results = {}
for depth in [3, None]:
    scores = []
    for seed in range(5):
        model = RandomForestClassifier(n_estimators=30, max_depth=depth, random_state=seed, n_jobs=1)
        model.fit(X_train, y_train)
        scores.append(model.score(X_val, y_val))
    results[str(depth)] = np.array(scores)
    print('depth=', depth, 'runs=', scores, 'mean=', np.mean(scores), 'sample std=', np.std(scores, ddof=1))
print('Paired validation differences:', results['None'] - results['3'])

## 如何解讀

比較的是驗證分數，並未讀取封存測試集的評估結果。選好設定後才進行最後測試。五個種子只提供有限的變異觀察，不能保證對所有資料切分都成立。

## 練習

將訓練樹數改為 10 或 100，先提出假設，再依相同種子集合比較，並記錄耗時。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **O9**：[OpenStax，§1.4 Experimental Design and Ethics](https://openstax.org/books/introductory-statistics-2e/pages/1-4-experimental-design-and-ethics)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
